In [17]:
import sys
import spikeinterface as si
import matplotlib.pyplot as plt
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.widgets as sw
import spikeinterface.qualitymetrics as sqm
import json
import probeinterface

from probeinterface import Probe, ProbeGroup

import os
import numpy as np
from spikeinterface.core import concatenate_recordings

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import pearsonr
import pandas as pd
import numpy as np
from matplotlib.collections import LineCollection
from probeinterface import write_probeinterface, read_probeinterface
import spikeinterface.exporters as sexp
from spikeinterface.core import write_binary_recording
from pathlib import Path
import pickle
from utils_clique import (
    CliqueInfo,
    build_shank_cliques,
    neuron_inf_dict_to_dataframe,
    get_recording_clique,
    filter_neuron_inf_by_clique,
    filter_gt_detect_array_by_clique,
    prepare_training_data,
    train_autosort_model
)


In [18]:
file_dict = {
    'mouse1': {
        1215: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse1_natima_251215_232216',
        1217: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse1_natima_251217_225054',
        1219: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse1&oldV1_natima_251219_201746'
    },
    'mouse2': {
        1214: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse&V1B_natima_251214_154409',
        1215: '/media/ubuntu/sda/mouse_test/raw_data/WLF_V1left&128ch2mouse_natima_251215_223556',
        1216: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse&V1left_natima_251216_214224',
        1217: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse2&V1left_natima_251217_220244',
        1218: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse2&V1od_natima_251218_214009',
        1219: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse2&V1left_natima_251219_192148'
    }

}

In [19]:
channel_list_A = ['A-000', 'A-001', 'A-002',
       'A-003', 'A-004', 'A-005', 'A-006', 'A-007', 'A-008', 'A-009',
       'A-010', 'A-011', 'A-012', 'A-013', 'A-014', 'A-015', 'A-016',
       'A-017', 'A-018', 'A-019', 'A-020', 'A-021', 'A-022', 'A-023',
       'A-024', 'A-025', 'A-026', 'A-027', 'A-028', 'A-029', 'A-030',
       'A-031', 'A-032', 'A-033', 'A-034', 'A-035', 'A-036', 'A-037',
       'A-038', 'A-039', 'A-040', 'A-041', 'A-042', 'A-043', 'A-044',
       'A-045', 'A-046', 'A-047', 'A-048', 'A-049', 'A-050', 'A-051',
       'A-052', 'A-053', 'A-054', 'A-055', 'A-056', 'A-057', 'A-058',
       'A-059', 'A-060', 'A-061', 'A-062', 'A-063', 'A-064', 'A-065',
       'A-066', 'A-067', 'A-068', 'A-069', 'A-070', 'A-071', 'A-072',
       'A-073', 'A-074', 'A-075', 'A-076', 'A-077', 'A-078', 'A-079',
       'A-080', 'A-081', 'A-082', 'A-083', 'A-084', 'A-085', 'A-086',
       'A-087', 'A-088', 'A-089', 'A-090', 'A-091', 'A-092', 'A-093',
       'A-094', 'A-095', 'A-096', 'A-097', 'A-098', 'A-099', 'A-100',
       'A-101', 'A-102', 'A-103', 'A-104', 'A-105', 'A-106', 'A-107',
       'A-108', 'A-109', 'A-110', 'A-111', 'A-112', 'A-113', 'A-114',
       'A-115', 'A-116', 'A-117', 'A-118', 'A-119', 'A-120', 'A-121',
       'A-122', 'A-123', 'A-124', 'A-125', 'A-126', 'A-127']

channel_list_B = ['B-000', 'B-001', 'B-002',
       'B-003', 'B-004', 'B-005', 'B-006', 'B-007', 'B-008', 'B-009',
       'B-010', 'B-011', 'B-012', 'B-013', 'B-014', 'B-015', 'B-016',
       'B-017', 'B-018', 'B-019', 'B-020', 'B-021', 'B-022', 'B-023',
       'B-024', 'B-025', 'B-026', 'B-027', 'B-028', 'B-029', 'B-030',
       'B-031', 'B-032', 'B-033', 'B-034', 'B-035', 'B-036', 'B-037',
       'B-038', 'B-039', 'B-040', 'B-041', 'B-042', 'B-043', 'B-044',
       'B-045', 'B-046', 'B-047', 'B-048', 'B-049', 'B-050', 'B-051',
       'B-052', 'B-053', 'B-054', 'B-055', 'B-056', 'B-057', 'B-058',
       'B-059', 'B-060', 'B-061', 'B-062', 'B-063', 'B-064', 'B-065',
       'B-066', 'B-067', 'B-068', 'B-069', 'B-070', 'B-071', 'B-072',
       'B-073', 'B-074', 'B-075', 'B-076', 'B-077', 'B-078', 'B-079',
       'B-080', 'B-081', 'B-082', 'B-083', 'B-084', 'B-085', 'B-086',
       'B-087', 'B-088', 'B-089', 'B-090', 'B-091', 'B-092', 'B-093',
       'B-094', 'B-095', 'B-096', 'B-097', 'B-098', 'B-099', 'B-100',
       'B-101', 'B-102', 'B-103', 'B-104', 'B-105', 'B-106', 'B-107',
       'B-108', 'B-109', 'B-110', 'B-111', 'B-112', 'B-113', 'B-114',
       'B-115', 'B-116', 'B-117', 'B-118', 'B-119', 'B-120', 'B-121',
       'B-122', 'B-123', 'B-124', 'B-125', 'B-126', 'B-127']

In [20]:
mouse_name = 'mouse2'
combined_output_base = f'/media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort/{mouse_name}'



In [21]:
# Clique级别训练流程（适配新的文件架构）
# 构建cliques
dates_list = [1214] 

probe = read_probeinterface('/media/ubuntu/sda/mouse_test/probe/tip_probe_128_1.json')
cliques = build_shank_cliques(probe, shank_boundaries=[250, 750, 1250])

# 对每个clique和每个date进行训练
for clique in cliques:
    clique_id = clique.clique_id
    print(f"\n{'='*60}")
    print(f"Processing Clique {clique_id}")
    print(f"{'='*60}")
    
    # 对每个date进行处理
    for date in dates_list:
        print(f"\n处理日期 {date}...")
        
        # 从新的文件架构读取数据
        date_data_folder = f'{combined_output_base}/clique_{clique_id}/date_{date}'
        neuron_inf_path = f'{date_data_folder}/neuron_inf.pickle'
        gt_detect_array_path = f'{date_data_folder}/gt_detect_array.csv'
        
        if not os.path.exists(neuron_inf_path) or not os.path.exists(gt_detect_array_path):
            print(f"  警告: {date_data_folder} 下没有找到数据文件，跳过")
            continue
        
        # 加载数据
        with open(neuron_inf_path, 'rb') as f:
            neuron_inf_dict = pickle.load(f)
        gt_detect_array = pd.read_csv(gt_detect_array_path)
        
        # 转换为DataFrame
        neuron_inf_date = neuron_inf_dict_to_dataframe(neuron_inf_dict)
        
        print(f"  Neurons: {len(neuron_inf_date)}")
        print(f"  Spikes: {len(gt_detect_array)}")
        
        # 为当前date读取并预处理recording（因为gt_detect_array的时间是相对时间）
        data_path = file_dict[mouse_name][date]
        file_list_path = Path(data_path)
        rhd_files = list(file_list_path.glob("*.rhd"))
        file_list = sorted(rhd_files)
        
        if len(file_list) == 0:
            print(f"  警告: 在 {data_path} 中未找到.rhd文件，跳过")
            continue
        
        # 读取并合并该date的所有rhd文件
        recording_raw_list = []
        for file in file_list:
            recording_raw_list.append(se.read_intan(file, stream_id='0'))
        
        date_recording = concatenate_recordings(recording_list=recording_raw_list)
        
        # 检测通道类型并选择对应的channel_list
        available_channels = date_recording.get_channel_ids()
        if 'A-127' in available_channels:
            channel_list = channel_list_A
        elif 'B-127' in available_channels:
            channel_list = channel_list_B
        else:
            print(f"  警告: 未找到A-127或B-127通道，跳过")
            continue
        
        # 选择通道
        date_recording = date_recording.select_channels(channel_list)
        
        # 统一将B开头的channel重命名为A开头
        channel_ids = date_recording.get_channel_ids()
        new_channel_ids = []
        renamed_count = 0
        for ch_id in channel_ids:
            if isinstance(ch_id, str) and ch_id.startswith('B-'):
                new_ch_id = 'A-' + ch_id[2:]
                new_channel_ids.append(new_ch_id)
                renamed_count += 1
            else:
                new_channel_ids.append(ch_id)
        
        if renamed_count > 0:
            date_recording = date_recording.rename_channels(new_channel_ids)
        
        # 预处理recording
        date_recording = spre.unsigned_to_signed(date_recording)
        date_recording = spre.resample(date_recording, 10000)
        date_recording = spre.bandpass_filter(date_recording, freq_min=300, freq_max=3000)
        date_recording = spre.notch_filter(date_recording, freq=50)
        date_recording = spre.common_reference(date_recording, reference="global", operator="median")
        date_recording = date_recording.set_probegroup(probe)
        
        # 获取recording_clique
        recording_clique = get_recording_clique(date_recording, clique)
        print(f"  Recording clique channels: {len(recording_clique.get_channel_ids())}")
        
        # 准备训练数据
        clique_save_dir = f'{combined_output_base}/clique_{clique_id}/date_{date}'
        train_data_dir = prepare_training_data(
            recording_f=recording_clique,
            gt_detect_array=gt_detect_array,
            neuron_inf=neuron_inf_date,
            save_dir=clique_save_dir,
            duration_seconds=1000,
            thr_min=2.5,
            thr_max=10,
            distance=3,
            wlen=5,
            prominence=15,
            left_sample=10,
            right_sample=20,
            max_firing_channel=None
        )
        
        # 训练模型（重复5次）
        n_channels = recording_clique.get_num_channels()
        n_repeats = 5
        
        for repeat_idx in range(1, n_repeats + 1):
            print(f"\n  ===== 重复训练 {repeat_idx}/{n_repeats} =====")
            model_save_dir = f'{clique_save_dir}/model_{repeat_idx}'
            
            autosort_model, training_log = train_autosort_model(
                train_data_dir=train_data_dir,
                model_save_dir=model_save_dir,
                n_channels=n_channels,
                left_sample=10,
                right_sample=20,
                epochs=20,
                batch_size=512,
                device=None,
                early_stopping=True,
                patience=5,
                min_delta=0.0,
                use_focal_loss=True,
                focal_gamma=2.0
            )
            
            print(f"  重复训练 {repeat_idx}/{n_repeats} 完成!")
        
        print(f"  Clique {clique_id}, Date {date} 所有重复训练完成!")

print("\n所有训练完成！")


[INFO] Shank 0: 32 channels (x range: min to 250)
[INFO] Shank 1: 32 channels (x range: 250 to 750)
[INFO] Shank 2: 32 channels (x range: 750 to 1250)
[INFO] Shank 3: 32 channels (x range: 1250 to max)
[INFO] Built 4 cliques from 4 shanks

Processing Clique 0

处理日期 1214...
  Neurons: 17
  Spikes: 340832
  Recording clique channels: 32
### 1. Threshold Detection
Sampling rate: 10000.0 Hz, Number of channels: 32
Recording total length: 15873472 samples (1587.35 seconds)
Will process first 10000000 samples (1000.00 seconds)
Data shape: (10000000, 32) (clique channels)
Using old detection method: extremum_channels
Using 17 valid channels from neuron extremum_channels
Building detect_array...
Number of detected spikes: 1643305
去重: 移除了115146个spikes（保留幅值更大的channel上的spike）
去重前: 1643305个spikes, 去重后: 1528159个spikes

### 2. Load Ground Truth and Match
Building gt_array from gt_detect_array...
GT spike count: 215529
---spike detection rate: 0.9449
Number of matched spikes: 203650
Number of unmatch

Extracting waveforms: 100%|██████████| 30/30 [00:30<00:00,  1.00s/it]


Waveform extraction completed!
waveform shape: (1528156, 32, 30)

### 4. Save Training Data
Save directory: /media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort/mouse2/clique_0/date_1214/train_data
  ✓ neuron_mapping.pkl saved
Saving data...
  ✓ X_waveform.pkl saved
  ✓ Y_spike_id.pkl saved
  ✓ Y_spike_id_noise.pkl saved
  ✓ X_spiketrain_time.pkl saved

All data saved to: /media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort/mouse2/clique_0/date_1214/train_data
Data statistics:
  - Total spike count: 1528156
  - Number of channels: 32
  - Window length: 30
  - Number of unique units: 17
  - Noise spike count: 1324506
  - Valid spike count: 203650

  ===== 重复训练 1/5 =====
Using device: cuda
Create dataset...
Dataset loaded:
  - Total samples: 1528156
  - Number of channels: 32
  - Window length: 30
  - Number of unique units: 17
  - Noise samples: 1324506.0
  - Non-noise samples: 203650.0
Model parameters:
  - Number of channels: 32
  - Window length: 30
  - Number of units: 1

Training: 100%|██████████| 2388/2388 [00:15<00:00, 153.91it/s]


epoch : 1/20, detection loss = 14.453323, classification loss = 558.904109


...

epoch : 18/20, val detection loss = 8.686750, classification loss = 20.131106
epoch : 18/20, val acc noise = 0.9689, val acc label = 0.9810
epoch : 19/20


Training: 100%|██████████| 4567/4567 [00:27<00:00, 163.34it/s]


epoch : 19/20, detection loss = 1.369697, classification loss = 20.511005


Validation: 100%|██████████| 1142/1142 [00:05<00:00, 212.14it/s]


epoch : 19/20, val detection loss = 8.990661, classification loss = 19.415420
epoch : 19/20, val acc noise = 0.9687, val acc label = 0.9820
Early stopping triggered at epoch 19
Best model was at epoch 14 with val_loss = 27.723649

Dataset split:
  - Training set: 2337979 samples
  - Validation set: 584495 samples
Final model saved
Training log saved to: /media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort/mouse2/clique_3/date_1214/model_5/training_log.csv
  重复训练 5/5 完成!
  Clique 3, Date 1214 所有重复训练完成!

所有训练完成！


In [16]:
# ============================================================
# 绘制每个clique的特征UMAP图
# ============================================================
# 每个clique生成一个PDF，包含4张UMAP图：
# 1. Noise detection GT
# 2. Noise detection predicted
# 3. Label classifier GT
# 4. Label classifier predicted

from umap import UMAP
import torch
from torch.utils import data
from tqdm import tqdm
from matplotlib.backends.backend_pdf import PdfPages

print("="*60)
print("绘制每个clique的特征UMAP图")
print("="*60)

# 重新导入utils_clique以确保使用最新的代码定义
import importlib
import utils_clique
importlib.reload(utils_clique)
SimpleAutoSort = utils_clique.SimpleAutoSort
SimpleWaveformLoader = utils_clique.SimpleWaveformLoader

dates_list = [1214]

for clique in cliques:
    clique_id = clique.clique_id
    print(f"\n{'='*60}")
    print(f"处理Clique {clique_id}")
    print(f"{'='*60}")
    
    for date in dates_list:
        date_data_folder = f'{combined_output_base}/clique_{clique_id}/date_{date}'
        train_data_dir = f'{date_data_folder}/train_data/'
        model_save_dir = f'{date_data_folder}/model/'
        
        # 检查文件是否存在
        if not os.path.exists(train_data_dir):
            print(f"  警告: {train_data_dir} 不存在，跳过")
            continue
        
        # 加载classification_mapping
        classification_mapping_path = f'{model_save_dir}/classification_mapping.pkl'
        if not os.path.exists(classification_mapping_path):
            print(f"  警告: {classification_mapping_path} 不存在，跳过")
            continue
        
        with open(classification_mapping_path, 'rb') as f:
            classification_mapping = pickle.load(f)
        keep_id_list = classification_mapping['label_list']
        
        n_channels = 32
        samplepoints = 30
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        # 创建模型
        autosort_model = SimpleAutoSort(
            ch_num=n_channels,
            samplepoints=samplepoints,
            device=device,
            set_shank_id=keep_id_list,
            save_dir=model_save_dir,
            pos_weight_noise=None,
            pos_weight_label=None
        )
        
        # 加载模型权重
        noise_model_path = f'{model_save_dir}/multitask_single_wave_clsfier_noise_clsfier.pth'
        label_model_path = f'{model_save_dir}/multitask_single_wave_clsfier_label_clsfier.pth'
        
        if not os.path.exists(noise_model_path) or not os.path.exists(label_model_path):
            print(f"  警告: 模型文件不存在，跳过")
            continue
        
        autosort_model.clsfier_noise.load_state_dict(torch.load(noise_model_path, map_location=device))
        autosort_model.clsfier_label.load_state_dict(torch.load(label_model_path, map_location=device))
        autosort_model.eval()
        
        print(f"  模型已加载")
        
        # 加载训练数据
        # 需要知道shank_channel，这里使用所有通道（0到31）
        shank_channel = list(range(n_channels))
        dataset = SimpleWaveformLoader(train_data_dir, shank_channel, Keep_id=keep_id_list)
        dataloader = data.DataLoader(dataset, batch_size=512, shuffle=False, num_workers=0)
        
        print(f"  数据集大小: {len(dataset)}")
        
        # 提取特征和预测
        all_noise_features = []  # Features for noise classifier (intermediate_forward)
        all_label_features = []  # Features for label classifier (intermediate_forward)
        all_noise_gt = []
        all_noise_pred = []
        all_label_gt = []
        all_label_pred = []
        
        print(f"  提取特征和预测...")
        with torch.no_grad():
            for batch_data in tqdm(dataloader, desc=f"Processing batches"):
                # SimpleWaveformLoader返回顺序: Img (n_channels, window_length), GT (unit one-hot), GT_binary (noise one-hot), Img_single, channel_index
                batch_Img, batch_unit_label_onehot, batch_noise_label_onehot, batch_single, batch_channel_indices = batch_data
                batch_Img = batch_Img.to(device)  # (batch_size, n_channels, window_length)
                batch_single = batch_single.to(device)  # (batch_size, window_length)
                batch_noise_label_onehot = batch_noise_label_onehot.to(device)
                batch_unit_label_onehot = batch_unit_label_onehot.to(device)
                batch_channel_indices = batch_channel_indices.to(device) if isinstance(batch_channel_indices, torch.Tensor) else torch.tensor(batch_channel_indices, device=device)
                
                # Flatten batch_Img to (batch_size, n_channels * window_length) for _prepare_input
                batch_size = batch_Img.shape[0]
                batch_multi = batch_Img.view(batch_size, -1)  # (batch_size, n_channels * window_length)
                
                # Prepare input: codes will be (batch_size, n_channels + 2, window_length)
                codes = autosort_model._prepare_input(batch_multi, batch_single, batch_channel_indices)
                
                # Noise classifier
                noise_features = autosort_model.clsfier_noise.intermediate_forward(codes)
                noise_output = autosort_model.clsfier_noise(codes)
                noise_pred = torch.argmax(noise_output, dim=1)  # (batch_size,)
                
                # Label classifier
                label_features = autosort_model.clsfier_label.intermediate_forward(codes)
                label_output = autosort_model.clsfier_label(codes)
                label_pred = torch.argmax(label_output, dim=1)  # (batch_size,)
                
                # 将one-hot标签转换为类别索引
                # batch_noise_label_onehot: (batch_size, 2) [noise, spike] -> 0=noise, 1=spike
                noise_gt = torch.argmax(batch_noise_label_onehot, dim=1)  # (batch_size,)
                # batch_unit_label_onehot: (batch_size, n_units) -> label index (如果全为0则label=-1表示noise)
                unit_label_gt = torch.argmax(batch_unit_label_onehot, dim=1)  # (batch_size,)
                # 如果one-hot全为0（即不在任何unit中），则argmax会返回0，需要检查是否真的是valid unit
                # 可以通过检查max值来判断：如果max值为0，则表示不是valid unit
                unit_label_valid = torch.max(batch_unit_label_onehot, dim=1)[0] > 0  # (batch_size,)
                unit_label_gt = torch.where(unit_label_valid, unit_label_gt, torch.tensor(-1, device=device))
                
                # 保存特征和标签
                all_noise_features.append(noise_features.cpu().numpy())
                all_label_features.append(label_features.cpu().numpy())
                all_noise_gt.append(noise_gt.cpu().numpy())
                all_noise_pred.append(noise_pred.cpu().numpy())
                all_label_gt.append(unit_label_gt.cpu().numpy())
                all_label_pred.append(label_pred.cpu().numpy())
        
        # 合并所有batch
        all_noise_features = np.concatenate(all_noise_features, axis=0)  # (n_samples, 30)
        all_label_features = np.concatenate(all_label_features, axis=0)  # (n_samples, 30)
        all_noise_gt = np.concatenate(all_noise_gt, axis=0)  # (n_samples,)
        all_noise_pred = np.concatenate(all_noise_pred, axis=0)  # (n_samples,)
        all_label_gt = np.concatenate(all_label_gt, axis=0)  # (n_samples,)
        all_label_pred = np.concatenate(all_label_pred, axis=0)  # (n_samples,)
        
        print(f"  特征提取完成:")
        print(f"    - Noise features shape: {all_noise_features.shape}")
        print(f"    - Label features shape: {all_label_features.shape}")
        print(f"    - Noise GT: {np.unique(all_noise_gt)}")
        print(f"    - Noise Pred: {np.unique(all_noise_pred)}")
        print(f"    - Label GT unique count: {len(np.unique(all_label_gt[all_label_gt >= 0]))}")
        print(f"    - Label Pred unique count: {len(np.unique(all_label_pred))}")
        
        # 限制样本数量以提高UMAP计算速度（如果数据太多）
        # 对于noise的UMAP，从全部数据中采样
        max_samples_for_noise_umap = 50000
        if len(all_noise_features) > max_samples_for_noise_umap:
            print(f"  数据量较大，随机采样 {max_samples_for_noise_umap} 个样本用于Noise UMAP")
            noise_indices = np.random.choice(len(all_noise_features), max_samples_for_noise_umap, replace=False)
            noise_features_for_umap = all_noise_features[noise_indices]
            noise_gt_for_umap = all_noise_gt[noise_indices]
            noise_pred_for_umap = all_noise_pred[noise_indices]
        else:
            noise_features_for_umap = all_noise_features
            noise_gt_for_umap = all_noise_gt
            noise_pred_for_umap = all_noise_pred
        
        # 对于label的UMAP，先从全部数据中筛选出spike（label >= 0），然后采样5000个点
        max_samples_for_label_umap = 30000
        spike_mask = all_label_gt >= 0  # 筛选出spike
        spike_indices = np.where(spike_mask)[0]
        
        if len(spike_indices) > max_samples_for_label_umap:
            print(f"  从 {len(spike_indices)} 个spike中随机采样 {max_samples_for_label_umap} 个样本用于Label UMAP")
            selected_spike_indices = np.random.choice(len(spike_indices), max_samples_for_label_umap, replace=False)
            label_indices = spike_indices[selected_spike_indices]
        else:
            print(f"  使用全部 {len(spike_indices)} 个spike用于Label UMAP")
            label_indices = spike_indices
        
        label_features_for_umap = all_label_features[label_indices]
        label_gt_for_umap = all_label_gt[label_indices]
        label_pred_for_umap = all_label_pred[label_indices]
        
        # 同时需要对应的noise预测结果，用于绘制label predicted图
        noise_pred_for_label_umap = all_noise_pred[label_indices]
        
        # UMAP降维
        print(f"  进行UMAP降维...")
        umap_noise = UMAP(n_components=2, random_state=42, n_neighbors=30, min_dist=0.1)
        umap_label = UMAP(n_components=2, random_state=42, n_neighbors=30, min_dist=0.1)
        
        noise_features_2d = umap_noise.fit_transform(noise_features_for_umap)
        label_features_2d = umap_label.fit_transform(label_features_for_umap)
        
        # 保存PDF
        pdf_path = f'{model_save_dir}/umap_visualization_clique_{clique_id}.pdf'
        print(f"  保存UMAP图到: {pdf_path}")
        
        with PdfPages(pdf_path) as pdf:
            # 1. Noise detection GT
            fig, ax = plt.subplots(1, 1, figsize=(6, 6))
            unique_labels = sorted(np.unique(noise_gt_for_umap))
            colors = ['lightgrey', 'orange']
            label_names = ['Noise', 'Spike']  # 通常0=noise, 1=spike
            
            for i, label in enumerate(unique_labels):
                mask = noise_gt_for_umap == label
                label_name = label_names[int(label)] if int(label) < len(label_names) else f'Class {int(label)}'
                ax.scatter(noise_features_2d[mask, 0], noise_features_2d[mask, 1], 
                          c=[colors[i]], label=label_name, alpha=1, s=1)
            
            ax.axis('off')
            plt.tight_layout()
            pdf.savefig(fig, bbox_inches='tight')
            plt.close()
            
            # 2. Noise detection Predicted
            fig, ax = plt.subplots(1, 1, figsize=(6, 6))
            unique_labels = sorted(np.unique(noise_pred_for_umap))
            colors = ['lightgrey', 'orange']
            
            for i, label in enumerate(unique_labels):
                mask = noise_pred_for_umap == label
                label_name = label_names[int(label)] if int(label) < len(label_names) else f'Class {int(label)}'
                ax.scatter(noise_features_2d[mask, 0], noise_features_2d[mask, 1], 
                          c=[colors[i]], label=label_name, alpha=1, s=1)
            
            ax.axis('off')
            plt.tight_layout()
            pdf.savefig(fig, bbox_inches='tight')
            plt.close()
            
            fig, ax = plt.subplots(1, 1, figsize=(4, 4))
            if len(label_gt_for_umap) > 0:
                valid_features = label_features_2d
                valid_labels = label_gt_for_umap
                
                unique_labels = sorted(np.unique(valid_labels))
                colors = plt.cm.tab20(np.linspace(0, 1, len(unique_labels)))
                
                for i, label in enumerate(unique_labels):
                    mask = valid_labels == label
                    # 将label索引映射回unit ID
                    if int(label) < len(keep_id_list):
                        unit_id = keep_id_list[int(label)]
                        label_name = f'Unit {unit_id}'
                    else:
                        label_name = f'Label {int(label)}'
                    ax.scatter(valid_features[mask, 0], valid_features[mask, 1], 
                              c=[colors[i]], label=label_name, alpha=1, s=1)
            
            ax.axis('off')
            plt.tight_layout()
            pdf.savefig(fig, bbox_inches='tight')
            plt.close()
            
            fig, ax = plt.subplots(1, 1, figsize=(4, 4))
            if len(label_pred_for_umap) > 0:
                valid_features = label_features_2d
                valid_labels = label_pred_for_umap
                
                unique_labels = sorted(np.unique(valid_labels))
                colors = plt.cm.tab20(np.linspace(0, 1, len(unique_labels)))
                
                for i, label in enumerate(unique_labels):
                    mask = valid_labels == label
                    # 将label索引映射回unit ID
                    if int(label) < len(keep_id_list):
                        unit_id = keep_id_list[int(label)]
                        label_name = f'Unit {unit_id}'
                    else:
                        label_name = f'Label {int(label)}'
                    ax.scatter(valid_features[mask, 0], valid_features[mask, 1], 
                              c=[colors[i]], label=label_name, alpha=1, s=1)
            ax.axis('off')
            plt.tight_layout()
            pdf.savefig(fig, bbox_inches='tight')
            plt.close()
        
        print(f"  PDF已保存: {pdf_path}")

print(f"\n{'='*60}")
print("所有UMAP图绘制完成")
print(f"{'='*60}")


绘制每个clique的特征UMAP图

处理Clique 0
  模型已加载
Dataset loaded:
  - Total samples: 1528156
  - Number of channels: 32
  - Window length: 30
  - Number of unique units: 17
  - Noise samples: 1324506.0
  - Non-noise samples: 203650.0
  数据集大小: 1528156
  提取特征和预测...


Processing batches: 100%|██████████| 2985/2985 [00:10<00:00, 285.73it/s]


  特征提取完成:
    - Noise features shape: (1528156, 30)
    - Label features shape: (1528156, 30)
    - Noise GT: [0 1]
    - Noise Pred: [0 1]
    - Label GT unique count: 17
    - Label Pred unique count: 17
  数据量较大，随机采样 50000 个样本用于Noise UMAP
  从 203650 个spike中随机采样 30000 个样本用于Label UMAP
  进行UMAP降维...
  保存UMAP图到: /media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort/mouse2/clique_0/date_1214/model//umap_visualization_clique_0.pdf
  PDF已保存: /media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort/mouse2/clique_0/date_1214/model//umap_visualization_clique_0.pdf

处理Clique 1
  模型已加载
Dataset loaded:
  - Total samples: 888883
  - Number of channels: 32
  - Window length: 30
  - Number of unique units: 12
  - Noise samples: 787295.0
  - Non-noise samples: 101588.0
  数据集大小: 888883
  提取特征和预测...


Processing batches: 100%|██████████| 1737/1737 [00:06<00:00, 275.34it/s]


  特征提取完成:
    - Noise features shape: (888883, 30)
    - Label features shape: (888883, 30)
    - Noise GT: [0 1]
    - Noise Pred: [0 1]
    - Label GT unique count: 12
    - Label Pred unique count: 12
  数据量较大，随机采样 50000 个样本用于Noise UMAP
  从 101588 个spike中随机采样 30000 个样本用于Label UMAP
  进行UMAP降维...
  保存UMAP图到: /media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort/mouse2/clique_1/date_1214/model//umap_visualization_clique_1.pdf
  PDF已保存: /media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort/mouse2/clique_1/date_1214/model//umap_visualization_clique_1.pdf

处理Clique 2
  模型已加载
Dataset loaded:
  - Total samples: 1104654
  - Number of channels: 32
  - Window length: 30
  - Number of unique units: 13
  - Noise samples: 1022923.0
  - Non-noise samples: 81731.0
  数据集大小: 1104654
  提取特征和预测...


Processing batches: 100%|██████████| 2158/2158 [00:07<00:00, 287.83it/s]


  特征提取完成:
    - Noise features shape: (1104654, 30)
    - Label features shape: (1104654, 30)
    - Noise GT: [0 1]
    - Noise Pred: [0 1]
    - Label GT unique count: 13
    - Label Pred unique count: 13
  数据量较大，随机采样 50000 个样本用于Noise UMAP
  从 81731 个spike中随机采样 30000 个样本用于Label UMAP
  进行UMAP降维...
  保存UMAP图到: /media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort/mouse2/clique_2/date_1214/model//umap_visualization_clique_2.pdf
  PDF已保存: /media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort/mouse2/clique_2/date_1214/model//umap_visualization_clique_2.pdf

处理Clique 3
  模型已加载
Dataset loaded:
  - Total samples: 2922474
  - Number of channels: 32
  - Window length: 30
  - Number of unique units: 31
  - Noise samples: 2734643.0
  - Non-noise samples: 187831.0
  数据集大小: 2922474
  提取特征和预测...


Processing batches: 100%|██████████| 5708/5708 [00:19<00:00, 291.03it/s]


  特征提取完成:
    - Noise features shape: (2922474, 30)
    - Label features shape: (2922474, 30)
    - Noise GT: [0 1]
    - Noise Pred: [0 1]
    - Label GT unique count: 31
    - Label Pred unique count: 31
  数据量较大，随机采样 50000 个样本用于Noise UMAP
  从 187831 个spike中随机采样 30000 个样本用于Label UMAP
  进行UMAP降维...
  保存UMAP图到: /media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort/mouse2/clique_3/date_1214/model//umap_visualization_clique_3.pdf
  PDF已保存: /media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort/mouse2/clique_3/date_1214/model//umap_visualization_clique_3.pdf

所有UMAP图绘制完成
